In [1]:
import os
import pandas as pd
from inference.classifier_predictor import ClassifierPredictor
from utils.dicom import load_image

# Leer el archivo Parquet
parquet_path = r"D:/TFM_CBIS/models/output/features/val_df_clean.parquet"
df = pd.read_parquet(parquet_path)

# Inicializar el modelo
strategy = strategies["XGBoost"]

if not SKIP_TRAINING:
    model_path = os.path.join(run_dir, f"xgboost_{correlation_id}.pkl")
else:
    correlation_id = "modelo_xgboost"
    model_path = os.path.join(MODELS_OUTPUT_PATH, correlation_id, f"xgboost_{correlation_id}.pkl")

predictor = ClassifierPredictor(strategy, model_path)

# Asumimos que hay una columna 'image_path' con rutas válidas a archivos de imagen
for i, row in df.iterrows():
    image_path = row["image_path"]  # Asegúrate de que esta columna exista

    try:
        image = load_image(image_path)
        result = predictor.predict(image)

        print(f"[{i}] Imagen: {os.path.basename(image_path)}")
        print(f"    → Predicción: {result['class_label']} (Confianza: {result['confidence']:.4f})")
        print(f"    → Probabilidades: {result['all_probs']}\n")

    except Exception as e:
        print(f"[ERROR] No se pudo procesar {image_path}: {e}")


ModuleNotFoundError: No module named 'inference'